# 03 - Train-only preprocessing

- **Meeting item:** Leakage control
- **Commit:** C05 - `feat: add train-only preprocessing contracts`
- **Commit achievement:** proves transforms are fitted on training cells only and then frozen.
- **Data mode:** `synthetic`
- **Evidence labels used:** verified, unknown
- **Human gate:** does the notebook show train-only statistics and rejection of validation or test identifiers during the fit?
- **Stop condition:** any validation or test value affects the fit.

All logic lives in `src/p22/data/transforms.py`. Order is fixed: split donors first, then fit.

In [1]:
import numpy as np
import pandas as pd

import p22
from p22.data import make_donor_split
from p22.data.transforms import PCA_KIND, fit_train_only
from p22.testing import make_synthetic_multimodal

dataset = make_synthetic_multimodal(seed=42)
split = make_donor_split(dataset.donor_ids, labels=dataset.labels, val_fraction=0.25, test_fraction=0.25, seed=0)

train_ids = dataset.cell_ids[split.train_index]
val_ids = dataset.cell_ids[split.val_index]
test_ids = dataset.cell_ids[split.test_index]
holdout_ids = np.concatenate([val_ids, test_ids])

print(f"data mode: {dataset.generation['data_mode']}")
print(f"train cells: {train_ids.size}, val cells: {val_ids.size}, test cells: {test_ids.size}")
print(f"donor overlap between train and holdout: {len(set(dataset.donor_ids[split.train_index]) & set(dataset.donor_ids[np.concatenate([split.val_index, split.test_index])]))}")

data mode: synthetic
train cells: 120, val cells: 60, test cells: 60
donor overlap between train and holdout: 0


## 1. Fit on training cells only (evidence label: verified)

The recorded statistics must equal the statistics of the training rows and nothing else.

In [2]:
scaler = fit_train_only(
    dataset.view_a,
    cell_ids=dataset.cell_ids,
    train_cell_ids=train_ids,
    holdout_cell_ids=holdout_ids,
)

train_mean = dataset.view_a[split.train_index].astype(np.float64).mean(axis=0)
all_cell_mean = dataset.view_a.astype(np.float64).mean(axis=0)

comparison = pd.DataFrame(
    {
        "fitted_mean": scaler.transformer.mean_[:5].round(4),
        "train_rows_mean": train_mean[:5].round(4),
        "all_cells_mean": all_cell_mean[:5].round(4),
    },
    index=[f"view_a_f{i}" for i in range(5)],
)
print(f"fitted statistics equal train-row statistics: {np.allclose(scaler.transformer.mean_, train_mean)}")
print(f"fitted statistics equal all-cell statistics: {np.allclose(scaler.transformer.mean_, all_cell_mean)}")
comparison

fitted statistics equal train-row statistics: True
fitted statistics equal all-cell statistics: False


,fitted_mean,train_rows_mean,all_cells_mean
view_a_f0,-1.0574,-1.0574,-1.4749
view_a_f1,-0.3254,-0.3254,0.1800
view_a_f2,-0.2560,-0.2560,0.1147
view_a_f3,-1.2995,-1.2995,-0.9755
view_a_f4,-0.0919,-0.0919,-1.1860


In [3]:
for key, value in scaler.metadata.items():
    if key == "parameters":
        print(f"parameters: {value}")
    else:
        print(f"{key}: {value}")

kind: standard_scaler
transformer_class: sklearn.preprocessing._data.StandardScaler
parameters: {'copy': True, 'with_mean': True, 'with_std': True}
training_id_hash: b78877874c3523e2b2dea831eb79fda475bf96ad2e837d829581ecf9ae7d6e84
n_train_cells: 120
n_holdout_ids_declared: 120
n_features_in: 16
n_features_out: 16
seed: 0
fitted_at: 2026-07-29T23:27:52+00:00
p22_version: 0.1.0
fit_scope: train cells only
parameter_fingerprint: 6da8274b7030a443981a973c4d923ac7a38db1f2bfeb68262d6d36b48f3479eb


## 2. Frozen for validation and test (evidence label: verified)

Training output is centred and scaled by construction. Held-out output is not, because the
same frozen parameters are applied to it. Held-out means at exactly zero would mean the
transform had seen held-out data.

In [4]:
rows = []
for name, positions in (("train", split.train_index), ("val", split.val_index), ("test", split.test_index)):
    transformed = scaler.transform(dataset.view_a[positions])
    rows.append(
        {
            "split": name,
            "cells": positions.size,
            "mean_abs_feature_mean": round(float(np.abs(transformed.mean(axis=0)).mean()), 5),
            "mean_feature_std": round(float(transformed.std(axis=0).mean()), 5),
        }
    )
pd.DataFrame(rows)

,split,cells,mean_abs_feature_mean,mean_feature_std
0,train,120,0.00000,1.00000
1,val,60,0.36313,0.98019
2,test,60,0.36027,0.95911


In [5]:
tampered = dataset.view_a.astype(np.float64).copy()
tampered[split.test_index] += 1000.0
tampered[split.val_index] *= -7.0

refit = fit_train_only(
    tampered,
    cell_ids=dataset.cell_ids,
    train_cell_ids=train_ids,
    holdout_cell_ids=holdout_ids,
)
print("held-out rows were multiplied and shifted, then the transform was refitted")
print(f"fingerprint unchanged: {refit.metadata['parameter_fingerprint'] == scaler.metadata['parameter_fingerprint']}")
print(f"training identifier hash unchanged: {refit.training_id_hash == scaler.training_id_hash}")

held-out rows were multiplied and shifted, then the transform was refitted
fingerprint unchanged: True
training identifier hash unchanged: True


## 3. Rejections (evidence label: verified)

In [6]:
attempts = {
    "validation identifier inside the fit set": lambda: fit_train_only(
        dataset.view_a,
        cell_ids=dataset.cell_ids,
        train_cell_ids=np.concatenate([train_ids, val_ids[:1]]),
        holdout_cell_ids=holdout_ids,
    ),
    "test identifier inside the fit set": lambda: fit_train_only(
        dataset.view_a,
        cell_ids=dataset.cell_ids,
        train_cell_ids=np.concatenate([train_ids, test_ids[:1]]),
        holdout_cell_ids=holdout_ids,
    ),
    "identifier not present in the matrix": lambda: fit_train_only(
        dataset.view_a,
        cell_ids=dataset.cell_ids,
        train_cell_ids=np.append(train_ids, "cell_99999"),
    ),
    "empty training set": lambda: fit_train_only(
        dataset.view_a, cell_ids=dataset.cell_ids, train_cell_ids=[]
    ),
    "wrong feature count at transform time": lambda: scaler.transform(dataset.view_b),
}

for description, attempt in attempts.items():
    try:
        attempt()
    except ValueError as error:
        print(f"rejected, {description}: {error}")
    else:
        raise AssertionError(f"not rejected: {description}")

rejected, validation identifier inside the fit set: validation or test identifiers appear in the fit set, which would leak held-out information: ['cell_00000']
rejected, test identifier inside the fit set: validation or test identifiers appear in the fit set, which would leak held-out information: ['cell_00020']
rejected, identifier not present in the matrix: train_cell_ids not present in cell_ids: ['cell_99999']
rejected, empty training set: train_cell_ids is empty; a transform needs training cells
rejected, wrong feature count at transform time: matrix has 12 features, transform was fitted on 16


In [7]:
mutable = fit_train_only(
    dataset.view_a,
    cell_ids=dataset.cell_ids,
    train_cell_ids=train_ids,
    holdout_cell_ids=holdout_ids,
)
mutable.transformer.mean_ = mutable.transformer.mean_ + 1.0
try:
    mutable.transform(dataset.view_a[split.test_index])
except ValueError as error:
    print(f"rejected, parameters mutated after fitting: {error}")
else:
    raise AssertionError("mutation was not detected")

rejected, parameters mutated after fitting: fitted parameters changed after fitting; transform is not frozen


## 4. Projection transform (evidence label: verified)

The same contract covers a projection. This is the step the legacy Tasic run did not have:
there the projection was fitted on all cells before splitting.

In [8]:
projection = fit_train_only(
    dataset.view_a,
    cell_ids=dataset.cell_ids,
    train_cell_ids=train_ids,
    holdout_cell_ids=holdout_ids,
    kind=PCA_KIND,
    n_components=5,
    seed=0,
)
print(f"kind: {projection.kind}")
print(f"features in: {projection.n_features_in} -> out: {projection.n_features_out}")
print(f"fitted on {projection.metadata['n_train_cells']} training cells")
print(f"test projection shape: {projection.transform(dataset.view_a[split.test_index]).shape}")
print(f"training projection mean near zero: {np.allclose(projection.transform(dataset.view_a[split.train_index]).mean(axis=0), 0.0, atol=1e-4)}")

kind: pca
features in: 16 -> out: 5
fitted on 120 training cells
test projection shape: (60, 5)
training projection mean near zero: True


In [9]:
assert dataset.generation["data_mode"] == p22.SYNTHETIC
assert np.allclose(scaler.transformer.mean_, train_mean)
assert not np.allclose(scaler.transformer.mean_, all_cell_mean)
assert scaler.metadata["n_train_cells"] == train_ids.size
assert scaler.metadata["fit_scope"] == "train cells only"
assert refit.metadata["parameter_fingerprint"] == scaler.metadata["parameter_fingerprint"]
test_transformed = scaler.transform(dataset.view_a[split.test_index])
assert np.abs(test_transformed.mean(axis=0)).max() > 1e-4
print("all C05 gate checks passed")

all C05 gate checks passed


## 5. Unknown (evidence label: unknown)

- Which transforms an approved dataset will need; only standardisation and a linear projection are covered here.
- Whether feature selection will be required. If it is, it must be fitted inside the training split under this same contract.

## Gate

**PASS**

- **Observed:** fitted statistics equal the training-row statistics and differ from all-cell statistics; training output is centred and scaled while held-out output is not; shifting and rescaling held-out rows leaves the fingerprint and training identifier hash unchanged; validation or test identifiers inside the fit set, unknown identifiers, an empty training set, a wrong feature count, and post-fit parameter mutation are all rejected; the same contract covers a five-component projection.
- **Remains unknown:** which transforms an approved dataset needs, and whether feature selection will be required.
- **Next decision owner:** researcher continues with C06 (metadata preflight).